In [11]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

from sklearn.metrics import mean_absolute_error, r2_score

import matplotlib.pyplot as plt
%matplotlib inline

# Hyper‐params
CSV_PATH    = "dataset_project_iapr2025/train.csv"
TRAIN_DIR   = "dataset_project_iapr2025/train/"
IMG_SIZE    = 160
BATCH_SIZE  = 32
LR          = 1e-3
WIDTH_MULT  = 1.0
N_EPOCHS    = 50
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cpu


In [12]:
class InvertedResidual(nn.Module):
    def __init__(self, in_c, out_c, stride, t):
        super().__init__()
        hidden = in_c * t
        self.use_res = (stride == 1 and in_c == out_c)
        layers = []
        if t != 1:
            layers += [
                nn.Conv2d(in_c, hidden, 1, bias=False),
                nn.BatchNorm2d(hidden), nn.ReLU6(inplace=True)
            ]
        layers += [
            nn.Conv2d(hidden, hidden, 3, stride, 1, groups=hidden, bias=False),
            nn.BatchNorm2d(hidden), nn.ReLU6(inplace=True),
            nn.Conv2d(hidden, out_c, 1, bias=False),
            nn.BatchNorm2d(out_c)
        ]
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return x + self.block(x) if self.use_res else self.block(x)

class MobileNetV2_Scratch(nn.Module):
    def __init__(self, n_classes, width_mult=1.0):
        super().__init__()
        inp = int(32 * width_mult)
        self.features = [nn.Conv2d(3, inp, 3, 2, 1, bias=False),
                         nn.BatchNorm2d(inp), nn.ReLU6(inplace=True)]
        cfgs = [(1,16,1,1),(6,24,2,2),(6,32,3,2),(6,64,4,2),
                (6,96,3,1),(6,160,3,2),(6,320,1,1)]
        for t,c,n,s in cfgs:
            out = int(c * width_mult)
            for i in range(n):
                stride = s if i==0 else 1
                self.features.append(InvertedResidual(inp, out, stride, t))
                inp = out
        last = int(1280 * width_mult) if width_mult>1.0 else 1280
        self.features += [nn.Conv2d(inp, last, 1, bias=False),
                          nn.BatchNorm2d(last), nn.ReLU6(inplace=True),
                          nn.AdaptiveAvgPool2d(1)]
        self.features = nn.Sequential(*self.features)
        self.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(last, n_classes))

    def forward(self, x):
        x = self.features(x).view(x.size(0),-1)
        return self.classifier(x)

# Instanciation
df = pd.read_csv(CSV_PATH)
classes = list(df.columns[1:])   # liste des noms de chocolats
n_classes = len(classes)

model = MobileNetV2_Scratch(len(df.columns)-1, width_mult=WIDTH_MULT).to(DEVICE)
print("Params:", sum(p.numel() for p in model.parameters()))


Params: 2240525


In [13]:
# ---- Métriques pour la régression de comptes
def mae(preds, target):
    # preds et target sont torch.Tensor CPU
    return mean_absolute_error(target.numpy(), preds.numpy())

def r2(preds, target):
    return r2_score(target.numpy(), preds.numpy())

# ---- Boucle d'un epoch avec logging métriques
def train_epoch(model, optimizer, criterion, metrics, loader, device):
    model.train()
    epoch_loss    = 0.0
    epoch_metrics = {k: 0. for k in metrics}
    
    for x_batch, y_batch in tqdm(loader, desc="Train", leave=False):
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        output = model(x_batch)
        loss   = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item() * x_batch.size(0)
        # calcul métriques
        with torch.no_grad():
            preds   = output.cpu()
            targets = y_batch.cpu()
            for name, fn in metrics.items():
                epoch_metrics[name] += fn(preds, targets)
    
    epoch_loss /= len(loader.dataset)
    for name in epoch_metrics:
        epoch_metrics[name] /= len(loader)
    
    print(f"Train → Loss: {epoch_loss:.4f}  " +
          "  ".join([f"{k}: {epoch_metrics[k]:.4f}" for k in epoch_metrics]))
    return epoch_loss, epoch_metrics


def plot_training(loss_log, metrics_names, metrics_log):
    n = len(metrics_names)+1
    fig, axes = plt.subplots(1, n, figsize=(5*n, 4))
    
    # loss
    axes[0].plot(loss_log, label="train")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("epoch")
    axes[0].legend()
    
    # each metric
    for i, name in enumerate(metrics_names, start=1):
        axes[i].plot(metrics_log[i-1], label="train")
        axes[i].set_title(name)
        axes[i].set_xlabel("epoch")
        axes[i].legend()
    
    plt.tight_layout()
    plt.show()


def update_metrics_log(metrics_names, log, new_dict):
    for i, name in enumerate(metrics_names):
        log[i].append(new_dict[name])
    return log


def train_cycle(model, optimizer, criterion, metrics, loader, n_epochs, device):
    loss_log = []
    names    = list(metrics.keys())
    metrics_log = [[] for _ in names]
    
    for epoch in range(1, n_epochs+1):
        print(f"=== Epoch {epoch}/{n_epochs} ===")
        loss, met = train_epoch(model, optimizer, criterion, metrics, loader, device)
        loss_log.append(loss)
        metrics_log = update_metrics_log(names, metrics_log, met)
        plot_training(loss_log, names, metrics_log)
    
    return loss_log, metrics_log


In [14]:
class ChocolateCountDataset(Dataset):
    def __init__(self, csv_path, img_dir, img_size=160, train=True):
        df = pd.read_csv(csv_path)
        self.ids     = df['id'].astype(str).tolist()
        self.counts  = df.iloc[:,1:].values.astype(np.float32)
        self.classes = list(df.columns[1:])
        self.img_dir = img_dir
        self.train   = train
        aug = [transforms.Resize((img_size,img_size)),
               transforms.RandomHorizontalFlip(),
               transforms.RandomRotation(20),
               transforms.ToTensor(),
               transforms.Normalize([0.485,0.456,0.406],
                                    [0.229,0.224,0.225])]
        noaug = [transforms.Resize((img_size,img_size)),
                 transforms.ToTensor(),
                 transforms.Normalize([0.485,0.456,0.406],
                                      [0.229,0.224,0.225])]
        self.transform = transforms.Compose(aug if train else noaug)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        img    = Image.open(os.path.join(self.img_dir, f"L{img_id}.JPG")).convert("RGB")
        x      = self.transform(img)
        y      = torch.from_numpy(self.counts[idx])
        return x, y

# instanciation + DataLoader
ds = ChocolateCountDataset(CSV_PATH, TRAIN_DIR, IMG_SIZE, train=True)
train_ds, _ = torch.utils.data.random_split(ds, [int(0.8*len(ds)), len(ds)-int(0.8*len(ds))])
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)


In [15]:
# instanciation
model     = MobileNetV2_Scratch(n_classes=len(ds.classes), width_mult=WIDTH_MULT).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR)
criterion = nn.MSELoss()  # ou L1Loss

# lancement
loss_log, metrics_log = train_cycle(
    model, optimizer, criterion,
    metrics={'MAE': mae, 'R2': r2},
    loader=train_loader,
    n_epochs=N_EPOCHS,
    device=DEVICE
)

# sauvegarde
torch.save(model.state_dict(), "choco_count_model.pth")


=== Epoch 1/50 ===


RuntimeError: DataLoader worker (pid(s) 15792, 19792, 5436, 14764) exited unexpectedly